# DUCX Quickstart — Colab & Local

> **DUCX: Decomposing Unfairness in Tool-Using Chest X-ray Agents**  
> Zikang Xu, Ruinan Jin, Xiaoxiao Li · [arXiv 2603.00777](https://arxiv.org/abs/2603.00777)

This notebook walks through the complete DUCX pipeline:

| Step | What happens |
|------|--------------|
| 1 | Clone the repo and install dependencies |
| 2 | Point to your data files |
| 3 | Configure your LLM endpoint |
| 4 | Dry-run to verify everything is wired up |
| 5 | Run the agent over the dataset |
| 6 | Run DUCX fairness decomposition analysis |
| 7 | Inspect results — summary, metrics, and figures |

---

### What you need before starting

- **GPU runtime** — in Colab: *Runtime → Change runtime type → T4 GPU*
- **LLM endpoint** — a running vLLM server (local or remote) **or** a Gemini API key for native Gemini tool-calling
- **Data** — MIMIC-FairnessVQA question JSONL + demographic CSV, or ChestAgentBench metadata (see `data/README.md`)
- **Judge API key** *(optional)* — for the LLM reasoning bias lens; the notebook uses DeepSeek by default but any OpenAI-compatible judge works

## Step 1 — Clone & Install

In **Colab** this cell clones the repository and installs the package.  
Running **locally from the repo root** it skips cloning and install automatically.

In [ ]:
# @title Repository setup
import importlib.util, os, subprocess, sys
from pathlib import Path

REPO_URL      = "https://github.com/Nanboy-Ronan/DUCK"  # @param {type:"string"}
BRANCH        = "main"                                   # @param {type:"string"}
INSTALL_PACKAGE = "auto"                                 # @param ["auto", "yes", "no"]

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    repo_name = Path(REPO_URL.rstrip("/").removesuffix(".git")).name or "DUCK"
    repo_dir  = Path("/content") / repo_name
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
else:
    if not Path("pyproject.toml").exists():
        raise RuntimeError("Run this notebook from the DUCK repository root, or open it in Colab.")

should_install = INSTALL_PACKAGE == "yes" or (INSTALL_PACKAGE == "auto" and IN_COLAB)
if should_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

print(f"Working directory : {Path.cwd()}")
print(f"Python            : {sys.executable}")
print(f"Colab             : {IN_COLAB}")

## Step 2 — Data Paths

Select your dataset and point the notebook at the local files.

- **MIMIC-FairnessVQA** — provided by the DUCX paper release. Place the full file at `data/mimic/medrax_input_all_2000.jsonl` and the demographic CSV at `data/mimic/mimic_sample_400.csv`.
- **ChestAgentBench** — metadata from [MedRAX](https://github.com/bowang-lab/MedRAX/tree/main/data); place at `data/chestagentbench/metadata.jsonl` and `data/eurorad_metadata.json`.

See `data/README.md` for the expected schema and download links.

In [ ]:
# @title Data configuration
from pathlib import Path

DATASET = "mimic"  # @param ["mimic", "chestagentbench"]

MIMIC_QUESTION_FILE          = "data/mimic/medrax_input_all_2000.jsonl"   # @param {type:"string"}
MIMIC_CASE_META              = "data/mimic/mimic_sample_400.csv"           # @param {type:"string"}
CHESTAGENTBENCH_QUESTION_FILE = "data/chestagentbench/metadata.jsonl"      # @param {type:"string"}
CHESTAGENTBENCH_CASE_META    = "data/eurorad_metadata.json"               # @param {type:"string"}

for d in ["data/chestagentbench", "data/mimic", "figures", "logs", "model-weights", "temp"]:
    Path(d).mkdir(parents=True, exist_ok=True)

if DATASET == "mimic":
    DATA_FILE         = MIMIC_QUESTION_FILE
    META_CASE_PATH    = MIMIC_CASE_META
    DEFAULT_LOG_PREFIX   = "qwen3vl8b-vllm-mimic"
    DEFAULT_ANALYSIS_DIR = "logs/mimic/analysis/qwen3vl8b-vllm/fairness_posthoc"
else:
    DATA_FILE         = CHESTAGENTBENCH_QUESTION_FILE
    META_CASE_PATH    = CHESTAGENTBENCH_CASE_META
    DEFAULT_LOG_PREFIX   = "qwen3vl8b-vllm"
    DEFAULT_ANALYSIS_DIR = "logs/chexagentbench/analysis/qwen3vl8b-vllm/fairness_posthoc"

print(f"Dataset        : {DATASET}")
print(f"Question file  : {DATA_FILE}  (exists: {Path(DATA_FILE).exists()})")
print(f"Case metadata  : {META_CASE_PATH}  (exists: {Path(META_CASE_PATH).exists()})")

## Step 3 — LLM Endpoint

**Option A — vLLM (recommended for reproducibility)**  
Start a local server before running this notebook, e.g.:
```bash
CUDA_VISIBLE_DEVICES=0 python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-VL-8B-Instruct --served-model-name qwen3-vl-8b \
  --port 8000 --enable-auto-tool-choice --tool-call-parser hermes
```
Then set `OPENAI_BASE_URL=http://localhost:8000/v1` and `OPENAI_MODEL=qwen3-vl-8b` below.

**Option B — Gemini native tool-calling**  
Set `USE_GEMINI_NATIVE=True` and add `GEMINI_API_KEY` to the Colab Secrets panel (🔑) before running.

In [ ]:
# @title LLM and run configuration
import os

OPENAI_BASE_URL    = "http://localhost:8000/v1"  # @param {type:"string"}
OPENAI_API_KEY     = "EMPTY"                     # @param {type:"string"}
OPENAI_MODEL       = "qwen3-vl-8b"               # @param {type:"string"}
USE_GEMINI_NATIVE  = False                        # @param {type:"boolean"}
DEVICE             = "cuda"                       # @param ["cuda", "cpu"]
MAX_CASES          = 0   # @param {type:"integer", description:"Limit cases for a quick trial. 0 = run all."}
LOG_PREFIX         = ""  # @param {type:"string",  description:"Leave blank to use the dataset default."}

os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
os.environ["OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["OPENAI_MODEL"]    = OPENAI_MODEL

LOG_PREFIX = LOG_PREFIX or DEFAULT_LOG_PREFIX

print(f"Model          : {OPENAI_MODEL}")
print(f"Endpoint       : {OPENAI_BASE_URL}")
print(f"Gemini native  : {USE_GEMINI_NATIVE}")
print(f"Device         : {DEVICE}")
print(f"Max cases      : {'all' if not MAX_CASES else MAX_CASES}")
print(f"Log prefix     : {LOG_PREFIX}")

## Step 4 — Verify Configuration (Dry Run)

This cell prints the exact launch command and runs it with `--dry-run` so the agent startup and tool loading are tested without sending any requests to the LLM. Fix any errors here before moving on.

In [ ]:
# @title Dry-run check
import subprocess, sys

launch_cmd = [
    sys.executable, "launch_over_chexbench.py",
    "--model",      OPENAI_MODEL,
    "--model-dir",  "./model-weights",
    "--temp-dir",   "./temp",
    "--data-file",  DATA_FILE,
    "--device",     DEVICE,
    "--log-prefix", LOG_PREFIX,
    "--llm-parse",
]
if USE_GEMINI_NATIVE:
    launch_cmd.append("--gemini-native")
if MAX_CASES and MAX_CASES > 0:
    launch_cmd += ["--max-cases", str(MAX_CASES)]

print("Launch command:")
print(" ".join(launch_cmd))
print()
subprocess.run(launch_cmd + ["--dry-run"], check=True)

## Step 5 — Run Agent Evaluation

Set `RUN_AGENT = True` once your data files, image paths, and LLM endpoint are ready.  
Logs are written to `logs/<log-prefix>/`. The path to the output JSON is printed when the run finishes — copy it into `LOG_PATH` in Step 6.

In [ ]:
# @title Launch agent evaluation
import glob

RUN_AGENT = False  # @param {type:"boolean"}

if not RUN_AGENT:
    print("Set RUN_AGENT = True to start evaluation.")
else:
    from pathlib import Path
    if not Path(DATA_FILE).exists():
        raise FileNotFoundError(f"Data file not found: {DATA_FILE}")
    subprocess.run(launch_cmd, check=True)

    # Report the log path for Step 6
    logs = sorted(glob.glob(f"logs/{LOG_PREFIX}/*.json"))
    if logs:
        print("\nAgent log written to:")
        for p in logs:
            print(" ", p)
        print("\nCopy the path above into LOG_PATH in Step 6.")
    else:
        print("No log file found under logs/{LOG_PREFIX}/")

## Step 6 — DUCX Fairness Analysis

Set `LOG_PATH` to the JSON log produced in Step 5, then set `RUN_ANALYSIS = True`.  
The LLM judge adds a reasoning-quality score to the output; enable it only when your judge API key is configured (see the Colab Secrets panel 🔑).  
Results land in `OUT_DIR` and are visualised automatically in Step 7.

In [ ]:
# @title DUCX fairness analysis
RUN_ANALYSIS       = False                          # @param {type:"boolean"}
LOG_PATH           = ""                             # @param {type:"string"}
OUT_DIR            = ""                             # @param {type:"string"}
ENABLE_LLM_JUDGE   = False                          # @param {type:"boolean"}
JUDGE_MODEL        = "deepseek-chat"                # @param {type:"string"}
JUDGE_BASE_URL     = "https://api.deepseek.com/v1" # @param {type:"string"}
JUDGE_API_KEY_ENV  = "DEEPSEEK_API_KEY"             # @param {type:"string"}

if not RUN_ANALYSIS:
    print("Set RUN_ANALYSIS = True after LOG_PATH points to a completed agent log.")
else:
    if not LOG_PATH:
        raise ValueError("Set LOG_PATH to the agent log produced in Step 5.")

    ANALYSIS_OUT = OUT_DIR or DEFAULT_ANALYSIS_DIR

    analysis_cmd = [
        sys.executable, "analysis/fairness_posthoc.py",
        "--log-path",      LOG_PATH,
        "--meta-q-path",   DATA_FILE,
        "--meta-case-path", META_CASE_PATH,
        "--out-dir",       ANALYSIS_OUT,
    ]
    if ENABLE_LLM_JUDGE:
        analysis_cmd += [
            "--enable-llm-judge",
            "--judge-model",       JUDGE_MODEL,
            "--judge-base-url",    JUDGE_BASE_URL,
            "--judge-api-key-env", JUDGE_API_KEY_ENV,
        ]

    print(" ".join(analysis_cmd))
    subprocess.run(analysis_cmd, check=True)
    print(f"\nOutputs written to: {ANALYSIS_OUT}")

## Step 7 — Inspect Results

The cells below read the analysis outputs and display them inline.  
Point `RESULT_DIR` at the directory used in Step 6 (defaults to `OUT_DIR` / `DEFAULT_ANALYSIS_DIR` automatically).

In [ ]:
# @title 7a — Summary report
from pathlib import Path
from IPython.display import Markdown, display

RESULT_DIR = Path(OUT_DIR or DEFAULT_ANALYSIS_DIR)

report = RESULT_DIR / "summary_report.md"
if report.exists():
    display(Markdown(report.read_text()))
else:
    print(f"summary_report.md not found in {RESULT_DIR}")
    print("Run Step 6 first, or set RESULT_DIR to the correct output directory.")

In [ ]:
# @title 7b — Group performance (accuracy & fairness metrics)
import pandas as pd

perf_path = RESULT_DIR / "group_performance.csv"
if perf_path.exists():
    perf = pd.read_csv(perf_path)
    display(perf.style.format(precision=3).background_gradient(cmap="RdYlGn", subset=pd.IndexSlice[:, perf.select_dtypes("number").columns]))
else:
    print(f"group_performance.csv not found in {RESULT_DIR}")

In [ ]:
# @title 7c — Tool exposure bias
import pandas as pd

gap_path = RESULT_DIR / "lens_agentic_conditional_tool_utility_gap.csv"
if gap_path.exists():
    gap = pd.read_csv(gap_path)
    display(gap.sort_values("utility_gap", key=abs, ascending=False).style.format(precision=3))
else:
    print(f"lens_agentic_conditional_tool_utility_gap.csv not found in {RESULT_DIR}")

In [ ]:
# @title 7d — Tool transition bias heatmap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

TOOL_LABELS = {
    "chest_xray_classifier": "CLS", "classifier": "CLS",
    "chest_xray_expert": "QA",      "llava_med_qa": "QA",   "xray_vqa": "QA",
    "chest_xray_report_generator": "RG", "report_generator": "RG",
    "chest_xray_segmentation": "SEG",    "segmentation": "SEG",
    "image_visualizer": "VIS",           "visualizer": "VIS",
    "xray_phrase_grounding": "GRD",      "phrase_grounding": "GRD",
    "START": "START",
}
MATRIX_TOOLS = ["CLS", "QA", "RG", "SEG", "VIS", "GRD"]
MATRIX_FROM  = ["START"] + MATRIX_TOOLS

trans_path = RESULT_DIR / "lens_agentic_transition_rates_by_group.csv"
if not trans_path.exists():
    print(f"lens_agentic_transition_rates_by_group.csv not found in {RESULT_DIR}")
else:
    trans = pd.read_csv(trans_path)
    trans[["from_tool", "to_tool"]] = trans["transition"].str.split("->", n=1, expand=True)
    trans["from_tool"] = trans["from_tool"].map(lambda x: TOOL_LABELS.get(str(x).strip(), str(x).strip()))
    trans["to_tool"]   = trans["to_tool"].map(lambda x: TOOL_LABELS.get(str(x).strip(), str(x).strip()))

    attributes = trans["attribute"].dropna().unique() if "attribute" in trans.columns else []
    n = len(attributes)
    if n == 0:
        print("No attribute columns found in transition data.")
    else:
        fig, axes = plt.subplots(1, n, figsize=(7 * n, 6), squeeze=False)
        for ax, attr in zip(axes[0], attributes):
            sub  = trans[trans["attribute"] == attr]
            groups = sub["subgroup"].dropna().unique() if "subgroup" in sub.columns else []
            if len(groups) < 2:
                ax.set_title(attr); ax.axis("off"); continue
            pos = sub[sub["subgroup"] == groups[0]]
            neg = sub[sub["subgroup"] == groups[1]]
            merged = pos.merge(neg, on=["from_tool", "to_tool"], suffixes=("_pos", "_neg"))
            merged["delta"] = merged["row_rate_pos"].astype(float) - merged["row_rate_neg"].astype(float)
            mat = (
                merged.pivot_table(index="from_tool", columns="to_tool", values="delta", aggfunc="mean")
                .reindex(index=MATRIX_FROM, columns=MATRIX_TOOLS)
                .fillna(0.0)
            )
            vmax = max(float(np.nanmax(np.abs(mat.to_numpy()))), 1e-6)
            sns.heatmap(
                mat, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                annot=True, fmt=".2f", annot_kws={"size": 9, "weight": "bold"},
                linewidths=0.5, linecolor="white", cbar=True,
            )
            title_label = attr.replace("gender", "gender: $P_{male}-P_{female}$").replace("age", "age: $P_{young}-P_{old}$")
            ax.set_title(f"Transition bias — {title_label}", fontweight="bold", fontsize=12)
            ax.set_xlabel(""); ax.set_ylabel("")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontweight="bold")
            ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontweight="bold")

        plt.tight_layout()
        plt.savefig(RESULT_DIR / "transition_bias_heatmap.png", dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
# @title 7e — LLM reasoning bias (optional, requires --enable-llm-judge)
import pandas as pd

reasoning_path = RESULT_DIR / "lens_llm_reasoning_bias.csv"
if reasoning_path.exists():
    reasoning = pd.read_csv(reasoning_path)
    display(reasoning.style.format(precision=3))
else:
    print(f"lens_llm_reasoning_bias.csv not found in {RESULT_DIR}")
    print("This file is generated only when --enable-llm-judge is passed in Step 6.")